# Day 16 Capstone: Customer Churn Prediction
End-to-end sklearn pipeline with leakage prevention, model comparison, tuning, evaluation, interpretation, and business recommendation.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

RNG = 42
rng = np.random.default_rng(RNG)
N = 4000
X_num, y = make_classification(n_samples=N, n_features=6, n_informative=5, n_redundant=0, n_clusters_per_class=1, weights=[0.76, 0.24], flip_y=0.13, class_sep=0.85, random_state=RNG)
Xs = (X_num - X_num.mean(0)) / X_num.std(0)
df = pd.DataFrame()
df['tenure_months'] = np.clip(Xs[:,4] * 12 + 34, 1, 72).round().astype(int)
df['monthly_charges'] = np.clip(Xs[:,0] * 13 + 72, 18, 130).round(2)
df['support_tickets'] = np.clip(Xs[:,2] * 1.7 + 3, 0, 25).round().astype(int)
df['avg_latency_ms'] = np.clip(Xs[:,3] * 35 + 120, 20, 400).round().astype(int)
df['addons_count'] = np.clip(Xs[:,1] * 1.3 + 4, 0, 8).round().astype(int)
df['days_since_login'] = np.clip(-Xs[:,4] * 8 + 16 + rng.normal(0,4,N), 0, 90).round().astype(int)
df['total_charges'] = (df['monthly_charges'] * df['tenure_months']).round(2)

def by_class(p_churn, p_stay, choices):
    out = np.empty(N, dtype=object); m = (y == 1)
    out[m] = rng.choice(choices, m.sum(), p=p_churn)
    out[~m] = rng.choice(choices, (~m).sum(), p=p_stay)
    return out

df['contract_type'] = by_class([0.75,0.18,0.07],[0.34,0.35,0.31], ['Month-to-month','One year','Two year'])
df['payment_method'] = by_class([0.55,0.20,0.15,0.10],[0.25,0.30,0.25,0.20], ['Electronic check','Mailed check','Bank transfer','Credit card'])
df['internet_service'] = by_class([0.65,0.28,0.07],[0.40,0.35,0.25], ['Fiber optic','DSL','None'])
df['has_dependents'] = by_class([0.82,0.18],[0.60,0.40], ['No','Yes'])
df['exit_survey_sent'] = np.where(y == 1, rng.binomial(1,0.97,N), rng.binomial(1,0.02,N))
df['churn'] = y
print('dataset shape:', df.shape, 'churn rate:', round(df.churn.mean(),3))
df.head()

## Step 1: Frame the problem and explore (EDA)
Predict whether a customer will churn (`churn = 1`) using account and plan features. Recall on the churn class is the business metric to protect, while ROC-AUC is used for threshold-independent model selection.

In [ ]:
print(df['churn'].value_counts(normalize=True).round(3), '\n')
num_cols = ['tenure_months','monthly_charges','support_tickets','avg_latency_ms','addons_count','days_since_login','total_charges']
print(df.groupby('churn')[num_cols].mean().round(1).T, '\n')
print(df.groupby('contract_type')['churn'].mean().round(3).sort_values(ascending=False))
fig, ax = plt.subplots(1,2,figsize=(11,4))
for k,lbl in [(0,'stay'),(1,'churn')]: ax[0].hist(df.loc[df.churn==k,'tenure_months'], bins=25, alpha=0.6, label=lbl)
ax[0].set_title('Tenure by outcome'); ax[0].set_xlabel('tenure_months'); ax[0].legend()
df.groupby('contract_type')['churn'].mean().sort_values().plot.barh(ax=ax[1])
ax[1].set_title('Churn rate by contract'); ax[1].set_xlabel('churn rate')
plt.tight_layout(); plt.show()

## Step 2: Split train / validation / test and detect leakage

In [ ]:
from sklearn.model_selection import train_test_split
leak_scores = df.corr(numeric_only=True)['churn'].drop('churn').abs().sort_values(ascending=False)
print('abs. correlation with churn:\n', leak_scores.round(3), '\n')
suspects = leak_scores[leak_scores > 0.9].index.tolist()
print('LEAK suspects:', suspects)
df_clean = df.drop(columns=suspects)
X = df_clean.drop(columns=['churn']); y = df_clean['churn']
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RNG)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RNG)
print('shapes:', X_train.shape, X_val.shape, X_test.shape)
print('churn rate train/val/test:', round(y_train.mean(),3), round(y_val.mean(),3), round(y_test.mean(),3))

## Step 3: Feature preprocessing inside a Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
num_features = X_train.select_dtypes('number').columns.tolist()
cat_features = X_train.select_dtypes('object').columns.tolist()
print('numeric:', num_features); print('categorical:', cat_features)
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
pre = ColumnTransformer([('num', num_pipe, num_features), ('cat', cat_pipe, cat_features)])
Z = pre.fit_transform(X_train)
print('transformed matrix:', Z.shape)
print('feature count after one-hot:', len(pre.get_feature_names_out()))

## Step 4: Train and compare four model families

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
models = {
 'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced'),
 'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RNG, class_weight='balanced'),
 'GradBoost': GradientBoostingClassifier(random_state=RNG),
 'SVM_RBF': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RNG)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
results = {}
for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    results[name] = scores
    print(f'{name:13s} ROC-AUC = {scores.mean():.3f} +/- {scores.std():.3f}')

## Step 5: Tune Gradient Boosting with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
gb_pipe = Pipeline([('pre', pre), ('clf', GradientBoostingClassifier(random_state=RNG))])
param_grid = {'clf__n_estimators':[150,300], 'clf__learning_rate':[0.05,0.1], 'clf__max_depth':[2,3]}
grid = GridSearchCV(gb_pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)
print('best params:', grid.best_params_)
print('best CV ROC-AUC:', round(grid.best_score_,3))
best_model = grid.best_estimator_
val_auc = roc_auc_score(y_val, best_model.predict_proba(X_val)[:,1])
print('validation ROC-AUC:', round(val_auc,3))

## Step 6: Final test evaluation and threshold choice
The test set is touched here for final evaluation. Threshold selection should normally be based on the validation set, so the notebook first inspects validation trade-offs and then reports final test performance at the selected threshold.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
val_proba = best_model.predict_proba(X_val)[:,1]
print('VALIDATION threshold trade-off')
print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'f1':>6}")
for t in [0.50,0.40,0.35,0.30,0.25]:
    p = (val_proba >= t).astype(int)
    print(f'{t:5.2f} {precision_score(y_val,p):10.3f} {recall_score(y_val,p):8.3f} {f1_score(y_val,p):6.3f}')
threshold = 0.30
proba = best_model.predict_proba(X_test)[:,1]
pred = (proba >= threshold).astype(int)
print('\nTEST ROC-AUC:', round(roc_auc_score(y_test, proba),3))
print(classification_report(y_test, pred, target_names=['stay','churn'], digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_test,pred), display_labels=['stay','churn']).plot(values_format='d')
plt.title(f'Confusion matrix @ threshold {threshold:.2f}'); plt.show()

## Step 7: Interpret the model

In [ ]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(best_model, X_val, y_val, n_repeats=8, random_state=RNG, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print('Permutation importance (ROC-AUC drop when shuffled):'); print(imp.round(4), '\n')
logreg = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))]).fit(X_train,y_train)
coef = pd.Series(logreg.named_steps['clf'].coef_[0], index=logreg.named_steps['pre'].get_feature_names_out())
print('Most churn-DRIVING (positive coef):\n', coef.sort_values().tail(4).round(2), '\n')
print('Most PROTECTIVE (negative coef):\n', coef.sort_values().head(4).round(2))
imp.sort_values().plot.barh(figsize=(7,4), title='Permutation importance'); plt.tight_layout(); plt.show()

## Step 8: Business recommendation
The churn model provides strong discrimination using only account and plan information available today. At an operating threshold of 0.30, the retention team can prioritize recall and identify a large share of customers at risk of leaving, while accepting some unnecessary contacts. The strongest drivers should be taken from the permutation-importance output above, with particular attention to tenure, contract characteristics, charges, and support activity when they rank highly. The retention team should prioritize high-risk customers for proactive support and test targeted incentives such as moving month-to-month customers toward longer contracts.

## Reflection
### 1. Leakage
Another possible leakage feature is `cancellation_date`, because it may only become available after a customer has decided to leave. I would check when every feature becomes available and investigate suspiciously high correlations or unrealistically high validation scores before trusting the model.

### 2. Metric choice
Precision would matter more if each positive prediction triggers an expensive retention action, such as a large discount or a costly outbound call. In that case, false positives waste substantial budget, so the company would want offers to go mainly to customers who really are likely to churn.

### 3. Trust
I would deploy the tuned Gradient Boosting pipeline because it performs strongly and can be interpreted using permutation importance. Since the candidate models achieve very similar ROC-AUC scores, switching algorithms is unlikely to create a large gain. The next improvement should therefore focus on better customer behaviour, service-quality, and support-interaction features.

### 4. Honesty
If the test set were used to choose the threshold, information from the final holdout set would influence the model decision. The reported test score would then be optimistic because the test set would no longer represent completely unseen data. Threshold selection should use validation data, with the test set reserved for the final evaluation.

In [ ]:
p30 = (proba >= 0.30).astype(int)
summary = {'primary_metric':'recall on churn (ROC-AUC for selection)', 'chosen_model':'GradientBoosting (tuned)', 'test_roc_auc':round(roc_auc_score(y_test,proba),3), 'operating_threshold':0.30}
summary['recall_at_0.30'] = round(recall_score(y_test,p30),3)
summary['precision_at_0.30'] = round(precision_score(y_test,p30),3)
summary['top_drivers'] = imp.head(3).index.tolist()
for k,v in summary.items(): print(f'{k:22}: {v}')